# Lesson 3.3 — Selecting and Filtering Rows/Columns

**Objectives**
- Select columns and rows with `[]`, `.loc`, and `.iloc`
- Filter rows with boolean masks and combine conditions (`&`, `|`)
- Sort and sample data

See `modules/03-pandas/notes.md` (Lesson 3.3) for the full written explanation.


In [1]:
import pandas as pd
from data_science_course.datasets import load_orders

orders = load_orders()
orders.head()

,order_id,customer_id,product_id,order_date,quantity,unit_price,discount_pct,payment_method,shipping_region,status,order_total
0,O000080,C00258,P0010,2023-07-30,1,76.29,0.05,gift_card,West,completed,72.48
1,O002751,C00559,P0007,2024-06-06,1,86.47,0.20,bank_transfer,West,completed,69.18
2,O003297,C00126,P0041,2023-10-22,1,268.04,0.25,credit_card,North,completed,201.03
3,O001169,C00272,P0040,2025-04-13,1,602.50,0.10,gift_card,East,completed,542.25
4,O003427,C00437,P0059,2025-02-08,1,540.06,0.10,paypal,South,completed,486.05


## Selecting columns

In [2]:
print(type(orders["order_total"]))       # Series
print(type(orders[["order_total"]]))     # DataFrame, one column
orders[["order_id", "order_total", "status"]].head(3)

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>


,order_id,order_total,status
0,O000080,72.48,completed
1,O002751,69.18,completed
2,O003297,201.03,completed


## `.loc` (by label) and `.iloc` (by position)

In [3]:
by_id = orders.set_index("order_id")
by_id.loc["O000001"]

customer_id            C00546
product_id              P0052
order_date         2022-09-09
quantity                    4
unit_price              14.78
discount_pct              0.0
payment_method         paypal
shipping_region          West
status              completed
order_total             59.12
Name: O000001, dtype: object

In [4]:
by_id.loc["O000001", "order_total"]

np.float64(59.12)

In [5]:
by_id.loc[["O000001", "O000002"], ["order_total", "status"]]

,order_total,status
order_id,,
O000001,59.12,completed
O000002,308.92,returned


In [6]:
print(orders.iloc[0])

order_id              O000080
customer_id            C00258
product_id              P0010
order_date         2023-07-30
quantity                    1
unit_price              76.29
discount_pct             0.05
payment_method      gift_card
shipping_region          West
status              completed
order_total             72.48
Name: 0, dtype: object


In [7]:
print(len(orders.iloc[0:3]))   # 3 rows: iloc end is EXCLUSIVE, like normal Python
print(len(by_id.loc[:by_id.index[2]]))  # loc end is INCLUSIVE of the end label

3
3


`.loc` slicing with `:` **includes** the end label; `.iloc` slicing **excludes** the
end position, exactly like ordinary Python list slicing. Mixing these up is a very
common source of off-by-one bugs -- when in doubt, check `len()` of the result.

## Boolean masks

In [8]:
mask = orders["status"] == "cancelled"
mask.head()

0    False
1    False
2    False
3    False
4    False
Name: status, dtype: bool

In [9]:
cancelled = orders[mask]
cancelled.shape

(193, 11)

In [10]:
orders.loc[orders["status"] == "cancelled", ["order_id", "customer_id", "order_total"]].head()

,order_id,customer_id,order_total
11,O005863,C00583,79.93
16,O000157,C00614,345.51
63,O001672,C00009,78.90
87,O001457,C00272,539.46
106,O003248,C00705,14.15


## Combining conditions

In [11]:
big_discount_returns = orders[
    (orders["status"] == "returned") & (orders["discount_pct"] > 0.15)
]
big_discount_returns.shape

(34, 11)

In [12]:
credit_or_paypal = orders[orders["payment_method"].isin(["credit_card", "paypal"])]
credit_or_paypal.shape

(2883, 11)

Try removing the parentheses around each condition above (`orders["status"] ==
"returned" & orders["discount_pct"] > 0.15`) and see what error you get -- `&` binds
tighter than `==`, so Python tries to evaluate `"returned" & orders["discount_pct"]`
first, which fails.

## Finding the 5 planted duplicate orders

In [13]:
dupe_mask = orders.duplicated(subset=["order_id"], keep="first")
orders[dupe_mask][["order_id", "customer_id", "product_id", "order_date", "quantity", "status"]]

,order_id,customer_id,product_id,order_date,quantity,status
1184,O001727,C00272,P0021,2024-06-25,2,completed
1538,O001457,C00272,P0041,2023-07-11,2,cancelled
2862,O004417,C00701,P0003,2023-06-30,1,completed
4430,O002848,C00514,P0001,2023-11-04,1,completed
5414,O001907,C00021,P0057,2024-08-11,1,completed


Exactly 5 rows -- matching the count documented in `data/README.md`. We'll remove
them for real with `.drop_duplicates()` in Lesson 3.4.

## Sorting and sampling

In [14]:
orders.sort_values("order_total", ascending=False).head(5)[["order_id", "order_total"]]

,order_id,order_total
3276,O002857,4105.10
3452,O001135,3460.05
3362,O004849,3309.95
1902,O000518,3273.55
4248,O002640,3165.60


In [15]:
orders.sort_values(["status", "order_total"], ascending=[True, False]).head(5)[["status", "order_total"]]

,status,order_total
2098,cancelled,2904.44
1611,cancelled,2708.46
741,cancelled,2352.84
5856,cancelled,1666.60
5692,cancelled,1577.66


In [16]:
orders.sample(5, random_state=42)[["order_id", "order_total"]]

,order_id,order_total
79,O003623,64.25
2750,O004927,117.56
3296,O000922,39.43
1168,O005040,336.56
3426,O001417,33.69


## Try it yourself

1. Select just the `order_id`, `product_id`, and `quantity` columns for orders where
   `quantity` is at least 4.
2. Using `.loc`, find all orders with `payment_method` equal to `"gift_card"` **and**
   `status` equal to `"returned"`. How many rows?
3. Sort `orders` by `order_date` ascending and show the 5 earliest orders'
   `order_id` and `order_date`.
4. Use `.duplicated()` with `keep=False` (instead of `"first"`) on `order_id` -- how
   many rows come back, and why is it different from the `keep="first"` result above?


In [17]:
# 1. TODO


# 2. TODO


# 3. TODO


# 4. TODO


### Solution

In [18]:
# 1.
big_orders = orders.loc[orders["quantity"] >= 4, ["order_id", "product_id", "quantity"]]
print(big_orders.shape)
print(big_orders.head())

# 2.
gift_returns = orders.loc[
    (orders["payment_method"] == "gift_card") & (orders["status"] == "returned")
]
print(gift_returns.shape[0])

# 3.
earliest = orders.sort_values("order_date", ascending=True).head(5)
print(earliest[["order_id", "order_date"]])

# 4.
all_dupe_rows = orders[orders.duplicated(subset=["order_id"], keep=False)]
print(all_dupe_rows.shape[0])
# keep=False marks BOTH the original AND every copy as a duplicate (10 rows: 5
# originals + 5 copies), while keep="first" only flags the extra copies (5 rows).

(294, 3)
    order_id product_id  quantity
30   O003169      P0023         4
54   O005958      P0007         4
84   O005033      P0033         5
94   O000394      P0049         4
101  O004568      P0054         5
100
     order_id  order_date
1914  O003015  2022-01-07
3299  O002805  2022-01-09
1637  O000567  2022-01-17
1200  O001506  2022-01-20
5321  O004469  2022-01-23
10
